# 02 - Dataset Analysis

This notebook performs Exploratory Data Analysis (EDA) on the MSMARCO-XI dataset. By understanding distribution properties like query length, passage count, and ground-truth balance, we can make informed engineering decisions about chunking sizes and embedding limits.

## Setup and Load Config

Load shared utilities and configuration.

In [ ]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in locals() else os.getcwd()))
colab_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if colab_root not in sys.path:
    sys.path.append(colab_root)

from src import utils, dataset_utils
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

utils.set_seed(42)
config = utils.load_config(os.path.join(colab_root, 'configs', 'experiment_config.yaml'))

## Load Dataset Sample

Loading the exact same configured sample used in Notebook 01 to run our analysis.

In [ ]:
LANGUAGE = 'hi'
SPLIT = 'validation'
N_SAMPLES = 5000

ds = dataset_utils.load_msmarco_xi(lang=LANGUAGE, split=SPLIT)
if N_SAMPLES and len(ds) > N_SAMPLES:
    ds = ds.select(range(N_SAMPLES))
print(f'Loaded {len(ds)} samples for analysis.')

## Analyze Lengths and Counts

We extract text lengths and passage counts for each example to understand token limits. We calculate character length for queries and passages.

In [ ]:
query_lengths = []
answer_lengths = []
passage_counts = []
passage_lengths = []
selected_passages_count = 0
non_selected_passages_count = 0

for item in ds:
    q = item.get('query', '')
    query_lengths.append(len(q))
    
    ans = item.get('answers', [])
    if ans:
        answer_lengths.append(len(ans[0]))
        
    passages = item.get('passages', [])
    passage_counts.append(len(passages))
    
    for p in passages:
        passage_lengths.append(len(p.get('passage_text', '')))
        if p.get('is_selected', 0) == 1:
            selected_passages_count += 1
        else:
            non_selected_passages_count += 1

print('Extraction complete.')

## Analyze Selected vs Non-Selected Passages

Understanding the class imbalance between relevant and irrelevant passages helps us design evaluation metrics like Recall@K vs MRR.

In [ ]:
print(f'Selected passages (Relevant): {selected_passages_count}')
print(f'Non-selected passages (Irrelevant): {non_selected_passages_count}')
try:
    ratio = selected_passages_count / (selected_passages_count + non_selected_passages_count)
    print(f'Ratio (Relevant to Total): {ratio:.4f}')
except ZeroDivisionError:
    print('Ratio (Relevant to Total): N/A')

## Analyze Query Types Distribution

Analyzing the distribution of query types (e.g., 'description', 'numeric') if the feature exists in the dataset.

In [ ]:
query_types = []
for item in ds:
    query_type = item.get('query_type')
    if query_type:
        query_types.append(query_type)

if query_types:
    qt_series = pd.Series(query_types)
    print('Query Types Distribution:')
    print(qt_series.value_counts())
else:
    print('Query type feature not found or empty.')

## Check for Duplicates, Empty, and Outliers

Checking if there are passages with zero length, or abnormally long passages that might cause memory out-of-bounds errors when embedding.

In [ ]:
empty_passages = sum(1 for l in passage_lengths if l == 0)
long_passages = sum(1 for l in passage_lengths if l > 5000) # Arbitrary threshold for characters

print(f'Empty passages: {empty_passages}')
print(f'Unusually long passages (>5000 chars): {long_passages}')

## Generate Distribution Statistics

Calculating mean, median, standard deviation, min, max, and percentiles for query and passage lengths.

In [ ]:
def get_stats(data, name):
    if not data: return {}
    arr = np.array(data)
    return {
        'name': name,
        'mean': float(np.mean(arr)),
        'median': float(np.median(arr)),
        'std': float(np.std(arr)),
        'min': int(np.min(arr)),
        'max': int(np.max(arr)),
        'p90': float(np.percentile(arr, 90)),
        'p95': float(np.percentile(arr, 95)),
        'p99': float(np.percentile(arr, 99))
    }

stats_q = get_stats(query_lengths, 'Query Lengths')
stats_p = get_stats(passage_lengths, 'Passage Lengths')
print(stats_q)
print(stats_p)

## Plotting: Query Length Distribution

**Engineering Question:** Do our queries generally fit within standard model limits (e.g., 512 tokens)?
This histogram shows the distribution of query character lengths.

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(query_lengths, bins=50, kde=True)
plt.title('Query Length Distribution')
plt.xlabel('Length (Characters)')
plt.ylabel('Frequency')
plt.show()

## Plotting: Passage Length Distribution

**Engineering Question:** What chunk size should we use? Should we split passages or embed them whole?
This boxplot reveals outliers and the overall distribution of passage lengths.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=passage_lengths)
plt.title('Passage Length Box Plot')
plt.xlabel('Length (Characters)')
plt.show()

## Save Analysis Results

Exporting our derived statistics into a JSON report for later review or automated metric tracking.

In [ ]:
reports_dir = utils.get_reports_dir()
os.makedirs(reports_dir, exist_ok=True)

analysis_results = {
    'total_examples': len(ds),
    'selected_passages': selected_passages_count,
    'non_selected_passages': non_selected_passages_count,
    'empty_passages': empty_passages,
    'long_passages': long_passages,
    'statistics': {
        'queries': stats_q,
        'passages': stats_p
    }
}

utils.save_json(analysis_results, os.path.join(reports_dir, 'dataset_analysis.json'))
print('Saved analysis results to dataset_analysis.json')